In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import plotly.express as px
import plotly.io as pio

In [1]:
engine = create_engine("postgresql+psycopg2://yuvalgat@localhost:5432/postgres")

df = pd.read_sql("SELECT * FROM dwh.vehicle LIMIT 10", engine)
print(df.head())

       _id  mispar_rechev  shnat_yitzur  kvutzat_zihum  tozeret_nm  \
0  2334944        5700514          2006            NaN  MITSUBISHI   
1  2335823       46250404          2025           11.0         BMW   
2  2336071       36141404          2025            3.0     HYUNDAI   
3  2466528        5943563          2007            NaN     HYUNDAI   
4  2336757       44503103          2023           14.0         KIA   

     tokef_dt moed_aliya_lakvish sug_delek_nm                    load_ts  \
0  2026-11-29               None         דיזל 2026-03-09 15:23:00.098380   
1  2026-11-09         2025-11-01        בנזין 2026-03-09 15:23:00.098380   
2  2026-09-02         2025-09-01        בנזין 2026-03-09 15:23:00.098380   
3  2025-06-17               None        בנזין 2026-03-09 15:23:00.098380   
4  2026-06-07         2023-06-01        בנזין 2026-03-09 15:23:00.098380   

   vehicle_age age_category  years_since_registration is_first_owner  \
0           20          Old                       

In [66]:
# Rendering in Jupyter
pio.renderers.default = "browser"   # אפשר גם "notebook_connected"

# Local PostgreSQL connection
DATABASE_URL = "postgresql://yuvalgat@localhost:5432/postgres"
engine = create_engine(DATABASE_URL)

# Fetch manufacturers
query_mfg = """
SELECT tozeret_nm AS manufacturer, COUNT(*) AS vehicle_count
FROM dwh.vehicle
WHERE tozeret_nm IS NOT NULL
GROUP BY tozeret_nm
ORDER BY vehicle_count DESC;
"""

df_mfg = pd.read_sql(query_mfg, engine)

# Calculate market share
total_vehicles = df_mfg["vehicle_count"].sum()
df_mfg["market_share_pct"] = (df_mfg["vehicle_count"] / total_vehicles) * 100

# Top 15
df_top15 = df_mfg.head(15).copy()
df_top15["label"] = df_top15["market_share_pct"].apply(lambda x: f"{x:.1f}%")

# Create chart
fig1 = px.bar(
    df_top15,
    x="manufacturer",
    y="vehicle_count",
    text="label",
    title="Market Share Analysis (Top 15 Manufacturers By Number Of Vehicles)",
    labels={"manufacturer": "Manufacturer", "vehicle_count": "Number of Vehicles"}
)

# Make all bars blue
fig1.update_traces(
    textposition="outside",
    textfont_size=12,
    marker_color="#1f77b4"
)

# Add subtitle + layout fixes
fig1.update_layout(
    xaxis_tickangle=-45,
    height=600,
    margin=dict(t=90, b=100),
    title=dict(
        text="Market Share Analysis (Top 15 Manufacturers By Number Of Vehicles)<br><sup>Percentage labels indicate each manufacturer's share of the total vehicle fleet.</sup>",
        x=0.5
    )
)

fig1.show()

In [68]:
# Rendering
pio.renderers.default = "browser"

# Local PostgreSQL connection
DATABASE_URL = "postgresql://yuvalgat@localhost:5432/postgres"
engine = create_engine(DATABASE_URL)

# Fetch data
query_age = """
SELECT age_category, COUNT(*) AS vehicle_count
FROM dwh.vehicle
WHERE age_category IS NOT NULL
GROUP BY age_category;
"""

df_age = pd.read_sql(query_age, engine)

# Order categories logically
category_order = ["New", "Recent", "Mature", "Old"]
fig2 = px.bar(
    df_age,
    x="age_category",
    y="vehicle_count",
    title="Fleet Age Distribution",
    labels={"age_category": "Age Category", "vehicle_count": "Number of Vehicles"},
    category_orders={"age_category": category_order},
    text_auto=".2s"
)

fig2.update_traces(
    textposition="outside",
    marker_color="#1f4e79",
    hoverinfo="skip"
)
fig2.show()

In [69]:
# Ensure proper rendering
pio.renderers.default = "browser"

# 1. Database Connection - LOCAL
DATABASE_URL = "postgresql://yuvalgat@localhost:5432/postgres"
engine = create_engine(DATABASE_URL)

# 2. Fetch data: Aggregate vehicle counts by production year and pollution level
query_env = """
SELECT shnat_yitzur AS production_year, pollution_level, COUNT(*) AS vehicle_count
FROM dwh.vehicle
WHERE shnat_yitzur >= 2000
  AND pollution_level IS NOT NULL
GROUP BY shnat_yitzur, pollution_level
ORDER BY shnat_yitzur ASC;
"""

df_env = pd.read_sql(query_env, engine)

# 3. Create the Stacked Area Chart
fig3 = px.area(
    df_env,
    x="production_year",
    y="vehicle_count",
    color="pollution_level",
    title="Environmental Trend (Production Year 2000+)",
    labels={
        "production_year": "Production Year",
        "vehicle_count": "Number of Vehicles",
        "pollution_level": "Pollution Level"
    },
    template="plotly_white",
    color_discrete_sequence=px.colors.qualitative.Set2
)

# 4. Layout adjustments
fig3.update_layout(
    xaxis=dict(
        dtick=2,
        range=[2000, df_env["production_year"].max()]
    ),
    yaxis_title="Total Vehicles",
    height=500,
    hovermode="x unified"
)

fig3.show()

In [70]:
# Ensure proper rendering
pio.renderers.default = "browser"

# 1. Database Connection - LOCAL POSTGRES
DATABASE_URL = "postgresql://yuvalgat@localhost:5432/postgres"
engine = create_engine(DATABASE_URL)

# 2. Fetch data: Aggregate vehicle counts by production year and fuel category
query_fuel_evo = """
SELECT shnat_yitzur AS production_year, fuel_category, COUNT(*) AS vehicle_count
FROM dwh.vehicle
WHERE shnat_yitzur >= 2000
  AND fuel_category IS NOT NULL
GROUP BY shnat_yitzur, fuel_category;
"""

df_fuel = pd.read_sql(query_fuel_evo, engine)

# 3. Create 5-year periods
df_fuel["period_start"] = (df_fuel["production_year"] // 5) * 5
df_fuel["period"] = (
    df_fuel["period_start"].astype(str) + "-" + (df_fuel["period_start"] + 4).astype(str)
)

# 4. Group by period + fuel category
df_grouped = (
    df_fuel.groupby(["period_start", "period", "fuel_category"], as_index=False)["vehicle_count"]
    .sum()
)

# 5. Calculate percentage distribution within each period
df_grouped["period_total"] = df_grouped.groupby("period")["vehicle_count"].transform("sum")
df_grouped["percent"] = (df_grouped["vehicle_count"] / df_grouped["period_total"]) * 100

# 6. Create labels only for visible segments
df_grouped["label"] = df_grouped["percent"].apply(lambda x: f"{x:.1f}%" if x >= 3.0 else "")

# 7. Rename latest period label for presentation
df_grouped["period_label"] = df_grouped["period"]
df_grouped.loc[df_grouped["period"] == "2025-2029", "period_label"] = "2025-Present"

# 8. Sort chronologically
df_grouped = df_grouped.sort_values("period_start")

# 9. Create the stacked bar chart
fig4 = px.bar(
    df_grouped,
    x="period_label",
    y="percent",
    color="fuel_category",
    text="label",
    custom_data=["vehicle_count"],
    title="Fuel distribution by 5-year periods",
    labels={
        "period_label": "Production Period",
        "percent": "Market Share (%)",
        "fuel_category": "Fuel Type"
    },
    template="plotly_white",
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig4.update_traces(
    textposition="inside",
    insidetextanchor="middle",
    hovertemplate="<b>Production period:</b> %{x}<br>"
                  "<b>Fuel type:</b> %{fullData.name}<br>"
                  "<b>Number of vehicles:</b> %{customdata[0]:,.0f}<br>"
                  "<b>Fleet share in this period:</b> %{y:.2f}%<extra></extra>"
)

fig4.update_layout(
    barmode="stack",
    yaxis=dict(title="Vehicle Share per Period (%)", range=[0, 100]),
    height=600,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.2,
        xanchor="center",
        x=0.5
    )
)

fig4.show()